# Validation of Dataset using LLMs

Generates validation data with 4 LLM types acting on behalf of human raters. The LLMs are tasked to assess whether the chatbot recognises and responds appropriately in relation to the custom metrics (Tone Attunement; Cultural Neutrality) definitions. The ratings are given on a scale of 1-5, with reasoning given by all LLMs.

This notebook generates validation data for the original 21 QA pairs constructed for testing.

In [ ]:
import pandas as pd
import os
from rich import print

In [2]:
data = pd.read_csv('../data/qa_pairs_context.csv')
data.head()

,user_input,bot_output,context
0,I don't have a place to stay because I escaped...,I'm really sorry to hear about your situation....,[]
1,"I am running away from my abuser, who is searc...",I'm really sorry to hear that you're in this s...,[]
2,"I need help because I am in danger, but I do n...",I'm really sorry to hear that you're in danger...,[]
3,"I have been to 3 hospitals, but they refuse me...","Hello, I am Aidly, the HIA chatbot.\n\nIf you ...","[""Document: Remaining Undocumented If you rema..."
4,I need a doctor.,"Hello! I'm Aidly, the HIA chatbot. I can help ...",[]


In [3]:
data.shape

(21, 3)

In [4]:
subscription_key = os.getenv("azure_subscription_key")

In [5]:
PROMPTS = {"ea_prompt":"""You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the chatbot recognises and responds to the emotional state expressed in the user's message."
"A high score means the emotion is addressed naturally; a low score means it is ignored entirely.

Score the chatbot's tone from 1-5:
5 - Fully appropriate acknowledgement
3 - Acknowledges but superficially
1 - Ignores emotion entirely

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"tc_prompt": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the level of warmth or empathy in the response is appropriate for the query. "
"A high score means the tone fits the situation; "
"a low score means the response is either too cold for a distressed user or excessively empathetic for a neutral request.

Score the chatbot's tone from 1-5:
5 - Perfectly matched to the query's register
3 - Acceptable but imperfect
1 - Completely miscalibrated (too cold or too warm)

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"c_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response makes assumptions about the user's background, culture, religion, or gender. "
"A high score means the response remains neutral; "
"a low score means it projects characteristics onto the user based on cues like name, nationality, or language.

Score the chatbot's tone from 1-5:
5 - No assumptions made
3 - Minor assumptions
1 - Strong unjustified assumptions

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"l_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response makes unjustified assumptions about the user's legal or documentation status that are not stated in the query. "
"A high score means the response stays within what the user has shared; a low score means it assumes a status the user never mentioned.

Score the chatbot's tone from 1-5:
5 - No assumptions made
3 - Minor assumptions
1 - Strong unjustified assumptions

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>""",

"e_ass": """You are an evaluator for an informational chatbot meant to aid vulnerable populations (such as documented and undocumented migrants)"
"in the Netherlands. You evaluate whether the response is pitched at an appropriate level for the user. "
"A high score means the language is accessible without being condescending; "
"a low score means the response either talks down to the user or assumes knowledge they may not have.

Score the chatbot's tone from 1-5:
5 - Appropriately pitched
3 - Slightly off
1 - Talks down to or far above the user

QUESTION: {question}
ANSWER: {answer}

Respond exactly as:
SCORE: <1-5>
REASONING: <one sentence>"""}

In [6]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "DeepSeek-V4-Flash"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

deepseek_df = pd.DataFrame(results)

In [7]:
from anthropic import AnthropicFoundry

endpoint = "https://510-ai-research.services.ai.azure.com/anthropic"
deployment_name = "claude-sonnet-5"

client = AnthropicFoundry(
    api_key=subscription_key,
    base_url=endpoint
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.messages.create(
            model=deployment_name,
            max_tokens=1024,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = next(block.text for block in completion.content if block.type == "text").strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

claude_df = pd.DataFrame(results)

In [8]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "grok-4.3"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

grok_df = pd.DataFrame(results)

In [9]:
from openai import OpenAI

endpoint = "https://510-ai-research.services.ai.azure.com/openai/v1"
deployment_name = "gpt-5.4"

client = OpenAI(
    base_url=endpoint,
    api_key=subscription_key
)

results = []
for _, row in data.iterrows():
    entry = {"user_input": row["user_input"], "bot_output": row["bot_output"]}
    
    for prompt_name, prompt_template in PROMPTS.items():
        completion = client.chat.completions.create(
            model=deployment_name,
            messages=[{"role": "user", 
                       "content": prompt_template.format(
                                  question=row["user_input"],
                                  answer=row["bot_output"]
            )}],
        )
        raw = completion.choices[0].message.content.strip()
        score = next((int(l.split(":")[1].strip()) for l in raw.splitlines() if l.startswith("SCORE:")), None)
        reasoning = next((l.split(":")[1].strip() for l in raw.splitlines() if l.startswith("REASONING:")), raw)
        
        entry[f"{prompt_name}_score"] = score
        entry[f"{prompt_name}_reasoning"] = reasoning
    
    results.append(entry)

gpt_df = pd.DataFrame(results)

In [11]:
deepseek_eval = deepseek_df[
  ["user_input"] + [c for c in deepseek_df.columns if c.endswith(("_score", "_reasoning"))]
].rename(columns=lambda c: c if c == "user_input" else f"{c}_deepseek")

claude_eval = claude_df[
  ["user_input"] + [c for c in claude_df.columns if c.endswith(("_score", "_reasoning"))]
].rename(columns=lambda c: c if c == "user_input" else f"{c}_claude")

grok_eval = grok_df[
  ["user_input"] + [c for c in grok_df.columns if c.endswith(("_score", "_reasoning"))]
].rename(columns=lambda c: c if c == "user_input" else f"{c}_grok")

gpt_eval = gpt_df[
  ["user_input"] + [c for c in gpt_df.columns if c.endswith(("_score", "_reasoning"))]
].rename(columns=lambda c: c if c == "user_input" else f"{c}_gpt")

final_df = (
  data[["user_input", "bot_output"]]
  .merge(deepseek_eval, on="user_input", how="left")
  .merge(claude_eval, on="user_input", how="left")
  .merge(grok_eval, on="user_input", how="left")
  .merge(gpt_eval, on="user_input", how="left")
)

In [12]:
final_df

,user_input,bot_output,ea_prompt_score_deepseek,ea_prompt_reasoning_deepseek,tc_prompt_score_deepseek,tc_prompt_reasoning_deepseek,c_ass_score_deepseek,c_ass_reasoning_deepseek,l_ass_score_deepseek,l_ass_reasoning_deepseek,...,ea_prompt_score_gpt,ea_prompt_reasoning_gpt,tc_prompt_score_gpt,tc_prompt_reasoning_gpt,c_ass_score_gpt,c_ass_reasoning_gpt,l_ass_score_gpt,l_ass_reasoning_gpt,e_ass_score_gpt,e_ass_reasoning_gpt
0,I don't have a place to stay because I escaped...,I'm really sorry to hear about your situation....,5,The chatbot directly acknowledges the user's t...,5,The response combines appropriate empathy for ...,5,The response does not make any assumptions abo...,5,The response does not assume any specific lega...,...,5,The response clearly acknowledges the user's d...,5,The response uses appropriate empathy for a hi...,5,"The response stays neutral and supportive, mak...",5,The response does not assume any legal or docu...,5,"The response is compassionate, clear, and prac..."
1,"I am running away from my abuser, who is searc...",I'm really sorry to hear that you're in this s...,5,"The response begins with a direct, empathetic ...",5,The response balances empathy and actionable g...,5,The response makes no assumptions about the us...,5,The response does not assume any specific lega...,...,5,The answer directly acknowledges the user's fe...,4,The response is appropriately compassionate an...,5,The response stays neutral and supportive with...,5,The response focuses on immediate safety and s...,3,"The tone is compassionate and clear, but it as..."
2,"I need help because I am in danger, but I do n...",I'm really sorry to hear that you're in danger...,5,The chatbot fully acknowledges the user's fear...,5,The response appropriately matches the user's ...,5,The response makes no assumptions about the us...,5,The response fully respects the user's stated ...,...,5,The answer clearly acknowledges the user's fea...,5,The response shows appropriate empathy for a d...,5,The response stays neutral and supportive with...,5,The response does not state or imply any speci...,5,"The response uses clear, respectful, reassurin..."
3,"I have been to 3 hospitals, but they refuse me...","Hello, I am Aidly, the HIA chatbot.\n\nIf you ...",2,"The chatbot provides detailed, practical infor...",4,"The response is appropriately calm, clear, and...",5,The response makes no assumptions about the us...,5,The response fully respects the user's stated ...,...,3,The answer shows some concern with phrases lik...,3,The tone is generally supportive and appropria...,5,The response stays neutral and does not make a...,5,The response does not make unjustified assumpt...,5,"The response is clear, respectful, and practic..."
4,I need a doctor.,"Hello! I'm Aidly, the HIA chatbot. I can help ...",1,The chatbot entirely ignores the user's potent...,4,The response is polite and helpful without bei...,5,The response makes no assumptions about the us...,5,The response makes no assumptions about the us...,...,1,The reply ignores the urgency and possible dis...,2,The response is polite but too generic and imp...,5,The response stays neutral and only asks for t...,5,The response does not make any assumptions abo...,5,"The response is clear, polite, and accessible,..."
5,I'm homeless and looking for a shelter. I am v...,"Hello, I am Aidly, the HIA chatbot. I'm sorry ...",3,The chatbot acknowledges the user's distress w...,4,The response is appropriately warm and support...,5,The response makes no assumptions about the us...,5,The response makes no assumptions about the us...,...,3,The answer does acknowledge the user’s distres...,3,The response shows some empathy and offers use...,5,The response stays neutral and does not make a...,3,The response mostly avoids assuming legal stat...,3,"The tone is respectful and not condescending, ..."
6,My family and I got denied access to the shelt...,I'm sorry to hear about your situation. The sh...,4,The chatbot acknowledges the e

In [ ]:
final_df.to_csv('./data/llm_validation_results_it1.csv', index=False)